### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model

/home/aniruddha/Projects/RAG/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ['OPENAI_API_KEY'] = os.getenv('OPENAI_API_KEY')
model = init_chat_model("openai:gpt-5-mini-2025-08-07")
response = model.invoke("What are fighter plane generations?")
response

AIMessage(content='"Fighter generations" is an informal way to group fighter aircraft by era and by the major technologies and capabilities they introduced. The divisions are not official and different sources draw the lines differently, but the common scheme runs from the earliest jet-era fighters up through today\'s stealthy types — and into a planned sixth generation. Below is a practical summary of the commonly used generations, their key characteristics, time frame, and typical examples.\n\nGeneral note\n- The classification emphasizes capabilities (engines/flight regime, weapons, sensors, avionics, stealth, networking, and mission roles) rather than strict dates.\n- Boundaries are fuzzy; many aircraft are transitional (often called "4.5‑generation") and different countries/analysts may label the same airplane differently.\n\nTypical generation breakdown\n\n1st generation (early jets / late WWII → early 1950s)\n- Era: late 1940s–early 1950s\n- Key features: first operational turbo

In [3]:
from langchain.tools import tool

@tool
def get_weather(location: str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

model_with_tools = model.bind_tools(
    [get_weather]
)

In [5]:
response_tool = model_with_tools.invoke(
    "What's the weather like in Mumbai?"
)

print(response_tool)

for tool_call in response_tool.tool_calls:
     # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 132, 'total_tokens': 155, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DxYaFYtKEUTWMntK1S7NTLag2Xyph', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019f283f-421f-7993-8d8b-561ba841e57b-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai'}, 'id': 'call_ogfzOJQc8e7QD2h2fIBshgp0', 'type': 'tool_call'}] usage_metadata={'input_tokens': 132, 'output_tokens': 23, 'total_tokens': 155, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Tool: get_weather
Args: {'location': 'Mumbai'}


### Tool Execution Loops

In [7]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Mumbai?"}]
ai_message = model_with_tools.invoke(messages)
messages.append(ai_message)

# Step 2: Execute tools and collect results
for tool_call in ai_message.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response.text)

It's sunny in Mumbai, India. Do you want more details (temperature, humidity, wind, forecast)?


In [8]:
messages

[{'role': 'user', 'content': "What's the weather in Mumbai?"},
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 131, 'total_tokens': 156, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DxZhpYatK4x5rdsuejsfzayslLv8O', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019f2881-1506-7e50-93fd-2f69cc862719-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Mumbai, India'}, 'id': 'call_S00b5agRQsOeIPDW6PWy7LqB', 'type': 'tool_call'}], usage_metadata={'input_tokens': 131, 'output_tokens': 25, 'total_tokens': 156, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {